📅 **论文年份 (Year):2014 年**  
*Recurrent Neural Network Regularization — Zaremba, Sutskever, Vinyals*

# Paper 4: Recurrent Neural Network Regularization(循环神经网络正则化)
## Wojciech Zaremba, Ilya Sutskever, Oriol Vinyals (2014)

### Dropout for RNNs(RNN 的 Dropout)

Key insight: Apply dropout to **non-recurrent connections only**, not recurrent connections.

核心洞见：只对**非循环连接（non-recurrent connections）**应用 dropout，而不对循环连接（recurrent connections）应用。

## 📖 论文导读

**🎯 这篇文章想解决什么问题（目的）：** 神经网络越大越容易"死记硬背"训练数据（即过拟合），而 dropout 是当时对付过拟合最好用的"药方"——训练时随机让一部分神经元"请假"，逼网络学得更扎实。但奇怪的是，这副药方用在处理序列数据的循环神经网络（RNN/LSTM）上却几乎失效，甚至帮倒忙。这篇论文要回答：dropout 到底该怎么用在 RNN 上？

**💡 主要贡献：** 作者找到了问题的症结：RNN 靠"循环连接"把记忆从上一时刻传到下一时刻，如果在这条记忆通道上随机丢弃信息，就像传话游戏里每一环都有人捣乱，传了几十步后早期的信息就被噪声冲垮了。解决办法出奇地简单——只在层与层之间的"竖直方向"用 dropout，绝不碰时间方向的循环连接。这让大型 LSTM 也能安全享受 dropout 的好处。

**🔧 方法：** 具体做法是：对输入到隐藏层、隐藏层到输出（以及多层 LSTM 层与层之间）的连接施加 dropout，而让隐藏状态在时间步之间原封不动地流动。作者在 Penn Treebank 语言建模等任务上验证，正确放置 dropout 后可以放心训练更大的网络，测试困惑度（perplexity，越低越好）从 78.4 降到 68.7。本笔记本还演示了后续的"变分 dropout"：整个序列共用同一张掩码，效果更稳定。

**🌟 意义：** 这篇论文让"大模型 + 强正则化"的配方在序列建模上跑通了，是此后几年 LSTM 统治机器翻译、语音识别、语言建模的重要基石。它也传递了一个朴素但深刻的经验：好技术不能生搬硬套，要先想清楚信息在网络里如何流动，再决定在哪里加约束。"保护记忆通路、只正则化其余部分"的思想，至今仍影响着各类序列模型的设计。

## 🎯 核心结论 (Key Takeaways)

- **论文核心发现：dropout 不能乱加，只能加在非循环连接上。** 对输入→隐藏（W_xh）和隐藏→输出（W_hy）这些"竖直方向"的连接用 dropout，而时间方向的循环连接（W_hh）必须原封不动——因为它是网络的记忆通道，每步都随机打断会把长序列的信息冲垮。

- **效果实打实：正确放置 dropout 后，大型 LSTM 在 Penn Treebank 语言建模上的测试困惑度从 78.4 降到 68.7**（perplexity 越低越好），证明"大模型 + 放对位置的正则化"这条路在序列模型上走得通。

- **本 notebook 用最小实现验证了机制**：`dropout` 函数演示了反向 dropout（p=0.5 时存活元素放大为 2 倍、均值不变，测试模式原样输出）；`RNNWithDropout` 的前向传播中，`W_hh·h` 一项刻意不加 dropout，与论文方案一一对应。

- **热力图对比直观展示了标准 vs 变分 dropout 的区别**：标准 dropout 每个时间步换一套掩码，隐藏状态图案杂乱；变分 dropout 整条序列共用同一套掩码，热力图呈整齐的竖直条纹（同一批单元从头到尾被屏蔽），时间维度上噪声更小、更稳定。

- **四张结构示意图总结了对错**："到处加 dropout"和"只在循环连接上加"都是错的（破坏时间信息流），只有 Zaremba 方案（输入端 + 输出端）既保住记忆又起到正则化作用。

- **带走一句话：先想清楚信息在网络里怎么流动，再决定在哪里加约束**——保护记忆通路、只正则化其余部分，这个思想至今仍影响着序列模型的设计。


## 🤯 反常识的发现 (Counterintuitive Findings)

- **常识认为：dropout 是"万能正则化"，哪里过拟合就往哪里加。** 但这篇论文发现，把 dropout 直接套在 RNN 上反而有害——循环连接（W_hh）是网络的记忆通道，每个时间步都随机丢一把，等于把时间方向的记忆也随机扔掉，传了几十步后早期信息就被噪声冲垮了。正确做法出人意料地"挑食"：只 drop 竖直方向的连接（输入→隐藏 W_xh、隐藏→输出 W_hy），放过水平方向的循环连接。本 notebook 的 `RNNWithDropout` 里 `W_hh·h` 一项刻意不加 dropout，正是这个道理。

- **常识认为：模型过拟合了，就该把模型改小一点。** 论文却反其道而行——把 dropout 放对位置之后，反而可以放心把 LSTM 做得更大：大型 LSTM 在 Penn Treebank 上的测试困惑度从 78.4 降到 68.7。"大模型 + 放对位置的正则化"胜过"小模型不正则化"，这条思路后来成了深度学习的主旋律。

- **常识认为：随机性越"新鲜"越好——每个时间步重新抽一套 dropout 掩码，噪声更丰富、正则化更充分。** 但后续的变分 dropout（notebook 里的 `RNNWithVariationalDropout`）发现恰恰相反：整条序列从头到尾共用同一套掩码，效果反而更稳定。对比热力图一眼就能看出区别：标准 dropout 每行（每个时间步）被清零的位置杂乱无章，变分 dropout 则是整列整列地"消失"——被屏蔽的神经元自始至终一致，不会每步给记忆通道注入新噪声。

- **常识认为：解决"dropout 在 RNN 上失效"这种难题，得发明复杂的新算法。** 实际的解法却近乎"零成本"：不改 dropout 本身、不改网络结构，只改它加在哪里。同样一个 dropout 函数，位置放错（如 notebook 四格示意图中"到处都加"或"只加循环连接"的两种错误方案）就帮倒忙，位置放对就立竿见影——代码上只是一两行的差别。


#### 💻 代码解读

**做什么:** 导入本笔记本需要的工具库，并固定随机种子，保证每次运行结果一致。

**怎么做:**
- 导入 `numpy`（数值计算库，负责矩阵运算）并简称为 `np`，导入 `matplotlib.pyplot`（画图库）简称 `plt`。
- 调用 `np.random.seed(42)` 固定随机数种子——就像掷骰子前先设定好点数序列，之后所有随机操作（比如 dropout 的随机遮罩）每次运行都会得到同样的结果，方便复现实验。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)

## Standard Dropout(标准 Dropout)

#### 💻 代码解读

**做什么:** 实现标准 dropout（随机失活）函数，并用一个全 1 向量做小实验演示它的效果。

**怎么做:**
- 定义 `dropout(x, dropout_rate, training)` 函数：训练时随机把一部分元素"抹成零"，测试时原样返回。就像上课时随机让一半学生闭嘴，逼所有学生都学会回答问题，防止网络"依赖个别神经元死记硬背"（过拟合）。
- 关键计算：用 `np.random.rand` 生成随机数并与 `dropout_rate` 比较，得到 0/1 的遮罩 `mask`；再除以 `(1 - dropout_rate)` 做放大补偿——这叫"反向 dropout"（inverted dropout），保证输出的平均值不变，测试时就不用再缩放。
- 若 `training=False` 或丢弃率为 0，直接返回原输入 `x`，即测试模式下 dropout 不起作用。
- 最后用一个 5 个元素全为 1 的向量 `x` 测试：打印两次 dropout 结果（每次被清零的位置不同，存活的值变成 2），以及测试模式下的结果（保持全 1）。

In [ ]:
def dropout(x, dropout_rate=0.5, training=True):
    """
    Standard dropout
    During training: randomly zero elements with probability dropout_rate
    During testing: scale by (1 - dropout_rate)
    """
    # 测试模式直接返回原值:反向dropout已在训练时完成缩放,推理时无需任何操作
    if not training or dropout_rate == 0:
        return x
    
    # Inverted dropout (scale during training)
    # *x.shape把形状元组解包成参数;比较运算得到布尔矩阵,再转成0/1浮点掩码
    mask = (np.random.rand(*x.shape) > dropout_rate).astype(float)
    # 除以(1-p)是"反向dropout":训练时放大保留的元素,使期望值不变,测试时就无需再缩放
    return x * mask / (1 - dropout_rate)

# Test dropout
x = np.ones((5, 1))
print("Original:", x.T)
print("With dropout (p=0.5):", dropout(x, 0.5).T)
print("With dropout (p=0.5):", dropout(x, 0.5).T)
print("Test mode:", dropout(x, 0.5, training=False).T)

## RNN with Proper Dropout(正确使用 Dropout 的 RNN)

**Key**: Dropout on **inputs** and **outputs**, NOT on recurrent connections!

**关键**：对**输入**和**输出**应用 dropout，而不是对循环连接（recurrent connections）！

#### 💻 代码解读

**做什么:** 实现论文（Zaremba 等人）提出的"正确用法"的 RNN：只在输入连接和输出连接上加 dropout，循环连接（h→h）绝不加。

**怎么做:**
- 定义 `RNNWithDropout` 类，`__init__` 里初始化三组权重矩阵：`W_xh`（输入→隐藏）、`W_hh`（隐藏→隐藏，即循环连接）、`W_hy`（隐藏→输出），外加偏置 `bh`、`by`，都乘 0.01 保持初始值很小。
- `forward` 方法逐个时间步处理输入序列：先对输入 `x` 调用上面定义的 `dropout` 函数得到 `x_dropped`（输入端加 dropout）。
- 用 `np.tanh(W_xh·x_dropped + W_hh·h + bh)` 更新隐藏状态 `h`——注意 `W_hh·h` 这一项没有加 dropout，因为循环连接是网络的"记忆通道"，若每步都随机打断，长期记忆会被破坏，就像接力赛每一棒都随机让选手摔倒，比赛就没法完成。
- 输出前再对隐藏状态 `h` 做一次 dropout 得到 `h_dropped`，然后计算 `y = W_hy·h_dropped + by`（输出端加 dropout）。
- 最后创建一个 10→20→10 的小 RNN，喂入 5 个随机输入，分别以训练模式和测试模式各跑一遍，打印两种模式下第一个输出的均值做对比。

In [ ]:
class RNNWithDropout:
    def __init__(self, input_size, hidden_size, output_size):
        self.input_size = input_size
        self.hidden_size = hidden_size
        self.output_size = output_size
        
        # Weights
        # 乘0.01做小随机初始化,防止tanh一开始就饱和;W_xh形状:(hidden, input)
        self.W_xh = np.random.randn(hidden_size, input_size) * 0.01
        self.W_hh = np.random.randn(hidden_size, hidden_size) * 0.01
        self.W_hy = np.random.randn(output_size, hidden_size) * 0.01
        self.bh = np.zeros((hidden_size, 1))
        self.by = np.zeros((output_size, 1))
    
    def forward(self, inputs, dropout_rate=0.0, training=True):
        """
        Forward pass with dropout
        
        Dropout applied to:
        1. Input connections (x -> h)
        2. Output connections (h -> y)
        
        NOT applied to:
        - Recurrent connections (h -> h)
        """
        # 初始隐状态h_0为零向量,形状:(hidden_size, 1)
        h = np.zeros((self.hidden_size, 1))
        outputs = []
        hidden_states = []

        for x in inputs:
            # Apply dropout to INPUT
            # 每个时间步重新采样一个dropout掩码(标准dropout的做法)
            x_dropped = dropout(x, dropout_rate, training)

            # RNN update (NO dropout on recurrent connection)
            # 论文核心:h->h的循环连接不加dropout,否则每步都丢信息,长期记忆会被破坏
            h = np.tanh(
                np.dot(self.W_xh, x_dropped) +  # Dropout HERE
                np.dot(self.W_hh, h) +           # NO dropout HERE
                self.bh
            )
            
            # Apply dropout to HIDDEN state before output
            # 只在h送往输出层的"非循环"路径上丢弃,h本身继续原样传给下一时刻
            h_dropped = dropout(h, dropout_rate, training)
            
            # Output
            y = np.dot(self.W_hy, h_dropped) + self.by  # Dropout HERE
            
            outputs.append(y)
            hidden_states.append(h)
        
        return outputs, hidden_states

# Test
rnn = RNNWithDropout(input_size=10, hidden_size=20, output_size=10)
test_inputs = [np.random.randn(10, 1) for _ in range(5)]

outputs_train, _ = rnn.forward(test_inputs, dropout_rate=0.5, training=True)
outputs_test, _ = rnn.forward(test_inputs, dropout_rate=0.5, training=False)

print(f"Training output[0] mean: {outputs_train[0].mean():.4f}")
print(f"Test output[0] mean: {outputs_test[0].mean():.4f}")

## Variational Dropout(变分 Dropout)

**Key innovation**: Use **same** dropout mask across all timesteps!

**关键创新**：在所有时间步（timestep）上使用**相同的** dropout 掩码（mask）！

#### 💻 代码解读

**做什么:** 实现"变分 dropout"（variational dropout）版本的 RNN：整条序列的所有时间步共用同一套 dropout 遮罩，而不是每一步重新抽签。

**怎么做:**
- 定义 `RNNWithVariationalDropout` 类，权重初始化与前面的 `RNNWithDropout` 完全一样。
- 核心区别在 `forward` 里：进入时间步循环**之前**，就一次性生成 `input_mask`（输入遮罩）和 `hidden_mask`（隐藏层遮罩），并顺手除以 `(1 - dropout_rate)` 做补偿；非训练模式则用全 1 遮罩（相当于不丢弃）。
- 循环中每个时间步都复用这同一套遮罩：`x_dropped = x * input_mask`、`h_dropped = h * hidden_mask`。打个比方：标准 dropout 是每节课随机点不同的学生"闭嘴"，变分 dropout 则是学期初就定好哪些学生整学期"闭嘴"——被屏蔽的神经元自始至终一致，时间维度上更稳定。
- 隐藏状态更新公式与之前相同（`tanh(W_xh·x_dropped + W_hh·h + bh)`），循环连接同样不加 dropout。
- 最后实例化 `var_rnn`，用之前的 `test_inputs` 以训练模式跑一遍，打印提示信息确认各时间步的遮罩保持一致。

In [ ]:
class RNNWithVariationalDropout:
    def __init__(self, input_size, hidden_size, output_size):
        self.input_size = input_size
        self.hidden_size = hidden_size
        self.output_size = output_size
        
        # Weights (same as before)
        self.W_xh = np.random.randn(hidden_size, input_size) * 0.01
        self.W_hh = np.random.randn(hidden_size, hidden_size) * 0.01
        self.W_hy = np.random.randn(output_size, hidden_size) * 0.01
        self.bh = np.zeros((hidden_size, 1))
        self.by = np.zeros((output_size, 1))
    
    def forward(self, inputs, dropout_rate=0.0, training=True):
        """
        Variational dropout: SAME mask for all timesteps
        """
        h = np.zeros((self.hidden_size, 1))
        outputs = []
        hidden_states = []
        
        # Generate masks ONCE for entire sequence
        # 变分dropout(Gal & Ghahramani):整条序列共用一份掩码,相当于每个样本采样一个"固定瘦身"的子网络
        if training and dropout_rate > 0:
            # 掩码已除以(1-p)完成反向缩放;形状:(input_size, 1),之后每步复用
            input_mask = (np.random.rand(self.input_size, 1) > dropout_rate).astype(float) / (1 - dropout_rate)
            hidden_mask = (np.random.rand(self.hidden_size, 1) > dropout_rate).astype(float) / (1 - dropout_rate)
        else:
            # 测试模式:全1掩码等价于不做dropout
            input_mask = np.ones((self.input_size, 1))
            hidden_mask = np.ones((self.hidden_size, 1))
        
        for x in inputs:
            # Apply SAME mask to each input
            # 与标准dropout的区别:不在循环内重新采样,每个时间步丢的都是同一批单元
            x_dropped = x * input_mask
            
            # RNN update
            h = np.tanh(
                np.dot(self.W_xh, x_dropped) +
                np.dot(self.W_hh, h) +
                self.bh
            )
            
            # Apply SAME mask to each hidden state
            h_dropped = h * hidden_mask
            
            # Output
            y = np.dot(self.W_hy, h_dropped) + self.by
            
            outputs.append(y)
            hidden_states.append(h)
        
        return outputs, hidden_states

# Test variational dropout
var_rnn = RNNWithVariationalDropout(input_size=10, hidden_size=20, output_size=10)
outputs_var, _ = var_rnn.forward(test_inputs, dropout_rate=0.5, training=True)

print("Variational dropout uses consistent masks across timesteps")

## Compare Dropout Strategies(比较不同的 Dropout 策略)

#### 💻 代码解读

**做什么:** 用热力图直观比较三种策略下隐藏状态的样子：不用 dropout、标准 dropout、变分 dropout。

**怎么做:**
- 先生成一条长度为 20 的随机测试序列 `test_sequence`（每步是 10 维随机向量）。
- 分别用三种方式跑前向传播并取回隐藏状态：`rnn.forward` 关掉 dropout 得到 `h_no_dropout`；`rnn.forward` 开 0.5 的 dropout 得到 `h_standard`；`var_rnn.forward` 开 0.5 的 dropout 得到 `h_variational`。
- 用 `np.hstack` 把每个时间步的隐藏向量拼成"时间步 × 隐藏单元"的二维矩阵，方便画图。
- 用 `plt.subplots(1, 3)` 画三张并排的热力图（`imshow`，红蓝配色），横轴是隐藏单元、纵轴是时间步。看图可发现：标准 dropout 每一行（每个时间步）被清零的位置都不一样，图案杂乱；变分 dropout 则是整列整列地"消失"（同一批单元从头到尾被屏蔽），呈现整齐的竖直条纹。

In [ ]:
# Generate synthetic sequence data
seq_length = 20
test_sequence = [np.random.randn(10, 1) for _ in range(seq_length)]

# Run with different strategies
# 用同一条输入序列对比三种策略,只关心隐状态(下划线丢弃outputs)
_, h_no_dropout = rnn.forward(test_sequence, dropout_rate=0.0, training=False)
_, h_standard = rnn.forward(test_sequence, dropout_rate=0.5, training=True)
_, h_variational = var_rnn.forward(test_sequence, dropout_rate=0.5, training=True)

# Convert to arrays
# 列表推导式:每个时间步的h形状为(hidden,1),flatten压成一维后用hstack首尾拼接
h_no_dropout = np.hstack([h.flatten() for h in h_no_dropout]).T
h_standard = np.hstack([h.flatten() for h in h_standard]).T
h_variational = np.hstack([h.flatten() for h in h_variational]).T

# Visualize
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 热力图:横轴隐单元、纵轴时间步;变分dropout应呈现整列一致的条纹(同一单元全程被丢)
axes[0].imshow(h_no_dropout, cmap='RdBu', aspect='auto')
axes[0].set_title('No Dropout')
axes[0].set_xlabel('Hidden Unit')
axes[0].set_ylabel('Time Step')

axes[1].imshow(h_standard, cmap='RdBu', aspect='auto')
axes[1].set_title('Standard Dropout (different masks per timestep)')
axes[1].set_xlabel('Hidden Unit')
axes[1].set_ylabel('Time Step')

axes[2].imshow(h_variational, cmap='RdBu', aspect='auto')
axes[2].set_title('Variational Dropout (same mask all timesteps)')
axes[2].set_xlabel('Hidden Unit')
axes[2].set_ylabel('Time Step')

plt.tight_layout()
plt.show()

print("Variational dropout shows consistent patterns (same units dropped throughout)")

## Dropout Placement Matters!(Dropout 的位置很重要！)

#### 💻 代码解读

**做什么:** 画四张 RNN 结构示意图，直观展示"dropout 加在哪里"是对是错，图解论文的核心结论。

**怎么做:**
- 定义辅助函数 `draw_rnn_cell(ax, title, ...)`：用 `plt.Rectangle` 画出四个方块——输入 `x_t`（浅蓝）、当前隐藏状态 `h_t`（浅绿）、上一步隐藏状态 `h_{t-1}`（浅黄）、输出 `y_t`（浅红），再用 `ax.arrow` 画三条连接箭头。
- 三个开关参数 `show_input_dropout`、`show_hidden_dropout`、`show_recurrent_dropout` 控制每条箭头的样式：加了 dropout 的连接画成红色粗箭头并标注红色"DROPOUT"字样，没加的画普通黑色细箭头，一眼就能看出 dropout 加在了哪里。
- 在 2×2 网格里画出四种方案：左上"错误——到处都加 dropout"（连循环连接也加，会打断时间信息流）；右上"错误——只在循环连接上加"（破坏梯度传递）；左下"正确——Zaremba 等人的方案"（只在输入和输出端加，循环连接保持畅通）；右下"基线——完全不加 dropout"（容易过拟合）。
- 最后 `plt.tight_layout()` 调整间距并 `plt.show()` 显示整幅对比图。

In [ ]:
# Visualize where dropout is applied
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# Create a simple RNN diagram
# 用三个布尔开关控制在哪条连接上画红色"DROPOUT"标记,直观对比不同放置策略
def draw_rnn_cell(ax, title, show_input_dropout, show_hidden_dropout, show_recurrent_dropout):
    ax.set_xlim(0, 10)
    ax.set_ylim(0, 10)
    ax.axis('off')
    ax.set_title(title, fontsize=12, fontweight='bold')
    
    # Draw boxes
    # Input
    ax.add_patch(plt.Rectangle((1, 2), 1.5, 1, fill=True, color='lightblue', ec='black'))
    ax.text(1.75, 2.5, 'x_t', ha='center', va='center', fontsize=10)
    
    # Hidden (current)
    ax.add_patch(plt.Rectangle((4, 4.5), 2, 2, fill=True, color='lightgreen', ec='black'))
    ax.text(5, 5.5, 'h_t', ha='center', va='center', fontsize=12)
    
    # Hidden (previous)
    ax.add_patch(plt.Rectangle((7, 4.5), 2, 2, fill=True, color='lightyellow', ec='black'))
    ax.text(8, 5.5, 'h_{t-1}', ha='center', va='center', fontsize=10)
    
    # Output
    ax.add_patch(plt.Rectangle((4, 7.5), 2, 1, fill=True, color='lightcoral', ec='black'))
    ax.text(5, 8, 'y_t', ha='center', va='center', fontsize=10)
    
    # Arrows
    # Input to hidden
    color_input = 'red' if show_input_dropout else 'black'
    width_input = 3 if show_input_dropout else 1
    ax.arrow(2.5, 2.5, 1.3, 2, head_width=0.3, color=color_input, lw=width_input)
    if show_input_dropout:
        ax.text(3.2, 3.5, 'DROPOUT', fontsize=8, color='red', fontweight='bold')
    
    # Recurrent
    color_rec = 'red' if show_recurrent_dropout else 'black'
    width_rec = 3 if show_recurrent_dropout else 1
    ax.arrow(7, 5.5, -0.8, 0, head_width=0.3, color=color_rec, lw=width_rec)
    if show_recurrent_dropout:
        ax.text(6.5, 6.2, 'DROPOUT', fontsize=8, color='red', fontweight='bold')
    
    # Hidden to output
    color_hidden = 'red' if show_hidden_dropout else 'black'
    width_hidden = 3 if show_hidden_dropout else 1
    ax.arrow(5, 6.6, 0, 0.7, head_width=0.3, color=color_hidden, lw=width_hidden)
    if show_hidden_dropout:
        ax.text(5.5, 7, 'DROPOUT', fontsize=8, color='red', fontweight='bold')

# Wrong: dropout everywhere
# 错误做法:循环连接也加dropout,记忆每步都被随机破坏,长序列信息无法保留
draw_rnn_cell(axes[0, 0], 'WRONG: Dropout Everywhere\n(Disrupts temporal flow)',
             show_input_dropout=True, show_hidden_dropout=True, show_recurrent_dropout=True)

# Wrong: only recurrent
draw_rnn_cell(axes[0, 1], 'WRONG: Only Recurrent\n(Loses gradient flow)', 
             show_input_dropout=False, show_hidden_dropout=False, show_recurrent_dropout=True)

# Correct: Zaremba et al.
# 论文结论:只在非循环连接(输入->隐层、隐层->输出)加dropout,既正则化又不伤记忆
draw_rnn_cell(axes[1, 0], 'CORRECT: Zaremba et al.\n(Input & Output only)',
             show_input_dropout=True, show_hidden_dropout=True, show_recurrent_dropout=False)

# No dropout
draw_rnn_cell(axes[1, 1], 'Baseline: No Dropout\n(May overfit)', 
             show_input_dropout=False, show_hidden_dropout=False, show_recurrent_dropout=False)

plt.tight_layout()
plt.show()

## Key Takeaways(要点总结)

### The Problem:(问题：)
- Naive dropout on RNNs doesn't work well
- Dropping recurrent connections disrupts temporal information flow
- Standard dropout changes mask every timestep (noisy)

- 在 RNN 上简单套用 dropout 效果不佳
- 丢弃循环连接会破坏时间信息流
- 标准 dropout 在每个时间步都更换掩码（噪声大）

### Zaremba et al. Solution:(Zaremba 等人的解决方案：)

**Apply dropout to:**
- ✅ Input-to-hidden connections (W_xh)
- ✅ Hidden-to-output connections (W_hy)

**Do NOT apply to:**
- ❌ Recurrent connections (W_hh)

**对以下连接应用 dropout：**
- ✅ 输入到隐藏层的连接 (W_xh)
- ✅ 隐藏层到输出的连接 (W_hy)

**不要对以下连接应用：**
- ❌ 循环连接 (W_hh)

### Variational Dropout:(变分 Dropout：)
- Use **same dropout mask** for all timesteps
- More stable than changing mask
- Better theoretical justification (Bayesian)

- 在所有时间步上使用**相同的 dropout 掩码**
- 比不断更换掩码更稳定
- 有更好的理论依据（贝叶斯，Bayesian）

### Results:(结果：)
- Significant improvement on language modeling
- Penn Treebank: Test perplexity improved from 78.4 to 68.7
- Works with LSTMs and GRUs too

- 在语言建模上取得显著提升
- Penn Treebank：测试困惑度（perplexity）从 78.4 改善到 68.7
- 同样适用于 LSTM 和 GRU

### Implementation Tips:(实现技巧：)
1. Use higher dropout rates (0.5-0.7) than feedforward nets
2. Apply dropout in **both** directions for bidirectional RNNs
3. Can stack multiple LSTM layers with dropout between them
4. Variational dropout: generate mask once per sequence

1. 使用比前馈网络更高的 dropout 率（0.5-0.7）
2. 对双向 RNN，在**两个**方向上都应用 dropout
3. 可以堆叠多层 LSTM，并在层与层之间加入 dropout
4. 变分 dropout：每个序列只生成一次掩码

### Why It Works:(为什么有效：)
- Preserves temporal dependencies (no dropout on recurrence)
- Regularizes non-temporal transformations
- Forces robustness to missing input features
- Consistent masks (variational) reduce variance

- 保留了时间依赖关系（循环连接上不加 dropout）
- 对非时间维度的变换进行正则化
- 迫使模型对缺失的输入特征保持鲁棒
- 一致的掩码（变分方式）降低了方差